# Phase 1: ClinVar Data Extraction & Cleaning
This notebook downloads the ClinVar `variant_summary.txt.gz`, processes it through several strict filtering steps to extract high-confidence missense variants, maps them to UniProt IDs, and validates the reference amino acids.

In [1]:
# !pip install pandas requests mygene biopython tqdm torch torchvision torchaudio
# !pip uninstall torch torchvision torchaudio -y
!python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://download.pytorch.org/whl/cu124


## 0. GPU Validation
Check if the GPU is active and available for PyTorch. This is crucial for the later GNN training phases.

In [2]:
import torch
if torch.cuda.is_available():
    print(f"GPU is Active and Available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is NOT available. PyTorch is using the CPU.")

GPU is Active and Available: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import pandas as pd
import numpy as np
import requests
import gzip
import io
import re
import time
from tqdm.notebook import tqdm
from Bio import SeqIO

## 1. Download ClinVar Variant Summary

We fetch the latest `variant_summary.txt.gz` from NCBI via **HTTPS** (not the raw FTP protocol).

> **Why HTTPS instead of FTP?**  
> The NCBI FTP server hosts the same file at both `ftp://` and `https://` paths. We use HTTPS because raw FTP is blocked by many corporate networks and cloud VM environments, making `ftp://` URLs unreliable for reproducibility. HTTPS is the standard for programmatic data access.

We verify the file size after downloading to ensure it was not truncated. The file is ~150-250 MB compressed and ~1.5 GB uncompressed.

**Data source:** Landrum et al., *Nucleic Acids Research*, 2014 - [ClinVar: public archive of relationships among sequence variation and human phenotype](https://doi.org/10.1093/nar/gkt1113)

In [4]:
# Using HTTPS instead of ftp:// for reliability across network environments
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
import urllib.request
import os

clinvar_file = "variant_summary.txt.gz"
if not os.path.exists(clinvar_file):
    print("Downloading ClinVar variant_summary.txt.gz... This may take a few minutes.")
    urllib.request.urlretrieve(CLINVAR_URL, clinvar_file)
    print("Download complete.")
else:
    print("File already exists locally. Skipping download.")

file_size_mb = os.path.getsize(clinvar_file) / (1024 * 1024)
print(f"File size on disk: {file_size_mb:.2f} MB")
if file_size_mb < 50:
    print("WARNING: File is unusually small. Download may have been truncated. Delete it and re-run.")
else:
    print("File size looks good (expected range: ~150–250 MB compressed).")

File already exists locally. Skipping download.
File size on disk: 419.14 MB
File size looks good (expected range: ~150–250 MB compressed).


## 2. Load Data

We load **only the 8 columns our pipeline actually uses** out of the 43 available columns.

> **Why not load all 43 columns?**  
> Loading all 43 columns of 8.3M rows as strings uses ~3-4 GB of RAM. On a local machine this causes the kernel to be killed by the OS mid-run (you see "The kernel appears to have died"). Loading only the required columns reduces peak memory usage by ~80%, from ~3.5 GB to ~500 MB.

| Column loaded | Used for |
|---|---|
| `VariationID` | Deduplication |
| `Type` | Filter: SNV only |
| `Name` | HGVS parsing → Ref_AA, Pos, Alt_AA |
| `GeneSymbol` | UniProt mapping |
| `ClinicalSignificance` | Label source (Target) |
| `ReviewStatus` | Confidence filter |
| `OriginSimple` | Filter: germline only |
| `Assembly` | Filter: GRCh38 only |

In [31]:
import gc

# Load only the 8 columns needed — not all 43.
# This reduces peak RAM from ~3.5 GB (all columns) to ~500 MB (selected only).
COLS_NEEDED = [
    'VariationID',           # unique identifier for deduplication
    'Type',                  # filter: keep SNV only
    'Name',                  # HGVS notation → parse Ref_AA, Pos, Alt_AA
    'GeneSymbol',            # map to UniProt → AlphaFold
    'ClinicalSignificance',  # source of binary label (Target)
    'ReviewStatus',          # confidence filter (2-star+)
    'OriginSimple',          # filter: keep germline only
    'Assembly',              # filter: keep GRCh38 only
]

print("Loading ClinVar data (selected columns only)...")
df = pd.read_csv(
    clinvar_file,
    sep='\t',
    compression='gzip',
    dtype=str,
    usecols=COLS_NEEDED,
    low_memory=False
)
gc.collect()   # immediately free any transient memory from the CSV parser

print(f"Raw variants loaded : {len(df):,}")
print(f"Columns in memory   : {list(df.columns)}")
print(f"Approx. memory used : {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

Loading ClinVar data (selected columns only)...
Raw variants loaded : 8,985,214
Columns in memory   : ['Type', 'Name', 'GeneSymbol', 'ClinicalSignificance', 'OriginSimple', 'Assembly', 'ReviewStatus', 'VariationID']
Approx. memory used : 4593 MB


## 3. Basic Filtering: Assembly, Variant Type, and Origin

Three pre-filters narrow the 8M+ raw ClinVar rows to the specific subset relevant to our structural GNN.

---

### 3a. GRCh38 Assembly Only
ClinVar stores each variant twice, once under GRCh37 (hg19) and once under GRCh38 (hg38). We keep only **GRCh38**, the current reference genome, for consistent genomic coordinates. All downstream tools (AlphaFold EBI, ESM-2, annotation databases) are aligned to GRCh38. Restricting to the current reference assembly is standard practice across all modern pathogenicity predictors - AlphaMissense (Cheng et al., 2023, [doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)) and gMVP (Ye et al., 2023, [doi:10.1038/s41467-023-42940-8](https://doi.org/10.1038/s41467-023-42940-8)) both operate on the current reference assembly.

---

### 3b. Single Nucleotide Variants (SNVs) Only
Our GNN models the effect of a **single amino acid substitution** at a known structural position. Insertions, deletions, copy number variants, and structural rearrangements alter the protein in fundamentally different ways (frameshift, splice disruption, domain loss) that require entirely different modelling strategies. We restrict to SNVs, consistent with REVEL ([doi:10.1016/j.ajhg.2016.08.016](https://doi.org/10.1016/j.ajhg.2016.08.016)), gMVP ([doi:10.1038/s41467-023-42940-8](https://doi.org/10.1038/s41467-023-42940-8)), and ClinPred ([doi:10.1016/j.ajhg.2018.05.013](https://doi.org/10.1016/j.ajhg.2018.05.013)).

---

### 3c. Germline Origin Only - `OriginSimple`, NOT `Origin`

> ⚠️ **Critical implementation detail:** We filter on `OriginSimple`, not `Origin`.

ClinVar provides two origin-related columns:

| Column | Description | Example values |
|---|---|---|
| `Origin` | Raw, submitter-provided string | `"germline"`, `"germline;somatic"`, `"maternal;germline"` |
| `OriginSimple` | ClinVar-curated, simplified single value | `"germline"`, `"somatic"`, `"unknown"`, `"not provided"` |

Using `Origin.str.contains("germline")` would incorrectly retain variants like `"germline;somatic"` - mutations reported in both inherited and cancer contexts. These follow different pathogenic mechanisms and introduce label ambiguity. `OriginSimple == "germline"` gives a clean, unambiguous filter.

Restricting to germline variants is standard in all inherited-disease pathogenicity predictors. AlphaMissense ([doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)) and ClinPred ([doi:10.1016/j.ajhg.2018.05.013](https://doi.org/10.1016/j.ajhg.2018.05.013)) both operate exclusively on germline missense variants from ClinVar.

In [32]:
# 3a. GRCh38 Assembly — keep current reference genome only
df = df[df['Assembly'] == 'GRCh38'].copy()
print(f"After GRCh38 filter:   {len(df):,}")

# 3b. SNV Type — keep single nucleotide variants only (no indels, CNVs, etc.)
df = df[df['Type'] == 'single nucleotide variant'].copy()
print(f"After SNV filter:      {len(df):,}")

# 3c. Germline Origin — use OriginSimple (ClinVar-curated), NOT Origin (raw submitter string)
# Origin can contain combined values like "germline;somatic" which are ambiguous.
# OriginSimple == "germline" gives a clean, single-context filter.
df = df[df['OriginSimple'] == 'germline'].copy()
print(f"After Germline filter: {len(df):,}")

After GRCh38 filter:   4,458,863
After SNV filter:      4,135,141
After Germline filter: 3,859,168


## 4. Clinical Significance & Confidence Filtering

---

### 4a. Remove Conflicting Interpretations First
ClinVar entries labelled `"Conflicting interpretations of pathogenicity"` reflect genuine scientific disagreement between submitting laboratories. The same variant is simultaneously called pathogenic by one lab and benign by another. Training on these introduces irreducibly ambiguous labels and degrades model calibration. We remove them **before** any label filter so they cannot slip through under a combined label string.

---

### 4b. Pathogenic / Benign Label Filter - Explicit Combined Labels Required

ClinVar does not use only four base labels. When multiple submitters classify a variant, ClinVar generates **compound label strings** that combine the individual classifications. A strict `.isin()` check against only the four base strings would silently drop many legitimate, high-confidence variants:

| ClinVar Label | Incorrectly dropped by base `.isin()`? | Correct Target |
|---|---|---|
| `Pathogenic` | No | 1 |
| `Likely pathogenic` | No | 1 |
| `Pathogenic/Likely pathogenic` | **Yes** | 1 |
| `Pathogenic, low penetrance` | **Yes** | 1 |
| `Likely pathogenic, low penetrance` | **Yes** | 1 |
| `Benign` | No | 0 |
| `Likely benign` | No | 0 |
| `Benign/Likely benign` | **Yes** | 0 |
| `Likely benign/Benign` | **Yes** | 0 |

We use **explicit label lists** covering all compound forms, consistent with the approach in AlphaMissense (Cheng et al., *Science* 2023, [doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)) and VARITY (Lin et al., *PLOS Genetics* 2022, [doi:10.1371/journal.pgen.1009423](https://doi.org/10.1371/journal.pgen.1009423)).

> **Label encoding:** `Target = 1` for pathogenic, `Target = 0` for benign. We use explicit list membership (`x in PATHOGENIC_LABELS`) rather than substring matching (`'pathogenic' in x.lower()`), which is fragile and can mislabel edge-case strings.

---

### 4c. ReviewStatus 2-Star Confidence Filter

ClinVar's `ReviewStatus` field encodes the confidence level of a variant's classification:

| ReviewStatus | Stars | We Keep? | Reason |
|---|---|---|---|
| `practice guideline` | ★★★★ | ✅ Yes | Expert-defined clinical guidelines |
| `reviewed by expert panel` | ★★★ | ✅ Yes | Consensus from domain specialists |
| `criteria provided, multiple submitters, no conflicts` | ★★ | ✅ Yes | ≥2 independent labs agree |
| `criteria provided, single submitter` | ★ | ❌ No | Unconfirmed - one lab only |
| `criteria provided, conflicting interpretations` | ★ | ❌ No | Labs disagree |
| `no assertion criteria provided` | - | ❌ No | No supporting evidence |

Retaining only 2-star and above ensures every training label has been independently confirmed by at least two laboratories with no disagreement. This directly reduces label noise, which is the primary driver of poor generalisation in pathogenicity models.

This filtering strategy is explicitly used by:
- **AlphaMissense** (Cheng et al., *Science* 2023) - [doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)
- **VARITY** (Lin et al., *PLOS Genetics* 2022) - [doi:10.1371/journal.pgen.1009423](https://doi.org/10.1371/journal.pgen.1009423)
- **gMVP** (Ye et al., *Nature Communications* 2023) - [doi:10.1038/s41467-023-42940-8](https://doi.org/10.1038/s41467-023-42940-8)

In [33]:
# 4a. Remove conflicting interpretations first — before any label filter
df = df[~df['ClinicalSignificance'].str.contains('Conflicting', na=False, case=False)].copy()
print(f"After removing Conflicting: {len(df):,}")

# 4b. Explicit label lists covering all ClinVar compound label strings.
# A simple .isin() with 4 base labels drops valid combined labels like
# "Pathogenic/Likely pathogenic" and "Benign/Likely benign".
PATHOGENIC_LABELS = [
    'Pathogenic',
    'Likely pathogenic',
    'Pathogenic/Likely pathogenic',
    'Pathogenic, low penetrance',
    'Likely pathogenic, low penetrance',
    'Likely pathogenic/Likely pathogenic, low penetrance',
    'Pathogenic/Likely pathogenic/Pathogenic, low penetrance',
    'Pathogenic/Likely pathogenic, low penetrance',
    'Pathogenic/Pathogenic, low penetrance',
    'Likely pathogenic/Pathogenic, low penetrance',
]
BENIGN_LABELS = [
    'Benign',
    'Likely benign',
    'Benign/Likely benign',
    'Likely benign/Benign',
]

df = df[df['ClinicalSignificance'].isin(PATHOGENIC_LABELS + BENIGN_LABELS)].copy()
print(f"After Pathogenic/Benign label filter: {len(df):,}")

# Explicit mapping — safer than substring matching ('pathogenic' in label.lower())
# which could mislabel edge-case strings.
df['Target'] = df['ClinicalSignificance'].apply(
    lambda x: 1 if x in PATHOGENIC_LABELS else 0
)
print(f"  Pathogenic (1): {(df['Target'] == 1).sum():,}")
print(f"  Benign     (0): {(df['Target'] == 0).sum():,}")

# 4c. ReviewStatus 2-star confidence filter — keep only high-confidence labels
valid_review_statuses = [
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'practice guideline',
]
df = df[df['ReviewStatus'].isin(valid_review_statuses)].copy()
print(f"After 2-star+ ReviewStatus filter: {len(df):,}")
print(f"  Pathogenic (1): {(df['Target'] == 1).sum():,}")
print(f"  Benign     (0): {(df['Target'] == 0).sum():,}")
print(f"  Class ratio: {(df['Target'] == 0).sum() / max((df['Target'] == 1).sum(), 1):.1f}:1 (benign:pathogenic)")

After removing Conflicting: 3,702,276
After Pathogenic/Benign label filter: 1,460,348
  Pathogenic (1): 175,402
  Benign     (0): 1,284,946
After 2-star+ ReviewStatus filter: 340,871
  Pathogenic (1): 60,265
  Benign     (0): 280,606
  Class ratio: 4.7:1 (benign:pathogenic)


## 5. Missense Extraction (HGVS Parsing)
We parse the `Name` column to ensure it contains a missense protein change (e.g., `p.Arg123Ser`). We drop synonymous (`=`) and nonsense (`*` or `Ter`) variants. We will also extract the Reference AA, Position, and Alternate AA for later validation.

In [34]:
def parse_missense(name_str):
    if pd.isna(name_str): return pd.Series({'Ref_AA': None, 'Pos': None, 'Alt_AA': None})
    match = re.search(r'\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|\*|Ter|=)\)', name_str)
    if match:
        ref_aa, pos, alt_aa = match.groups()
        if ref_aa != alt_aa and alt_aa not in ['Ter', '*', '=', 'Del', 'Dup']:
            return pd.Series({'Ref_AA': ref_aa, 'Pos': int(pos), 'Alt_AA': alt_aa})
    return pd.Series({'Ref_AA': None, 'Pos': None, 'Alt_AA': None})

parsed_cols = df['Name'].apply(parse_missense)
df = pd.concat([df, parsed_cols], axis=1)
df = df.dropna(subset=['Ref_AA']).copy()
df['Pos'] = df['Pos'].astype(int)
print(f"After Missense extraction: {len(df):,}")

After Missense extraction: 62,428


## 6. Deduplication & Multi-Gene Exclusion

---

### 6a. Deduplication by VariationID
ClinVar's `variant_summary.txt.gz` can contain multiple rows for the same `VariationID` due to multiple `AlleleID` records or re-submissions under different accessions. After filtering to GRCh38 only, we deduplicate to ensure each unique ClinVar entry contributes exactly one row. Without this step, the same variant could appear in both training and test splits, directly inflating performance metrics.

---

### 6b. Multi-Gene Exclusion
Some ClinVar entries annotate variants in overlapping genomic regions shared by two genes. These appear as semicolon- or comma-separated `GeneSymbol` values (e.g., `BRCA1;NBR2`). They are **structurally ambiguous**: we cannot unambiguously assign a single protein structure for GNN construction. Excluding them is consistent with the single-gene variant assumption used throughout the field - gMVP ([doi:10.1038/s41467-023-42940-8](https://doi.org/10.1038/s41467-023-42940-8)) and AlphaMissense ([doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)) both operate on variants with a single unambiguous gene assignment.

---

### 6c. Biological Deduplication (applied at end of Step 7, after UniProt mapping)
`VariationID` deduplication removes redundant ClinVar records. But after mapping to UniProt, a second issue emerges: different `VariationID`s may represent the **same biochemical event** - the same protein, same position, same amino acid substitution - if they were submitted under different transcript accessions or gene entries.

We resolve this with a second deduplication keyed on `(UniProt_ID, Pos, Alt_AA)`. This ensures each unique biological variant appears exactly once, preventing the model from learning the same example multiple times and preventing train/test contamination at the protein-variant level.

In [35]:
df = df.drop_duplicates(subset=['VariationID']).copy()
print(f"After VariationID dedup     : {len(df):,}")

df = df.dropna(subset=['GeneSymbol']).copy()
df = df[~df['GeneSymbol'].str.contains(';', na=False)].copy()
df = df[~df['GeneSymbol'].str.contains(',', na=False)].copy()
print(f"After multi-gene exclusion  : {len(df):,}")

# Free memory before the API calls — the dataset is now small enough
# that we no longer need to hold any discarded rows.
gc.collect()
print(f"Memory after filtering      : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

After VariationID dedup     : 62,412
After multi-gene exclusion  : 62,201
Memory after filtering      : 40.2 MB


## 7. UniProt Mapping: Swiss-Prot Only, via UniProt's Official ID Mapping API

We map each `GeneSymbol` to its canonical UniProt accession using **UniProt's own programmatic ID Mapping service** (`https://rest.uniprot.org/idmapping`). This is the same backend that powers the [UniProt Map IDs](https://www.uniprot.org/id-mapping) web tool - you can verify any result manually there.

This mapping is the bridge to two critical downstream steps:
1. **Phase 2 - AlphaFold structure download**: AlphaFold EBI indexes structures by Swiss-Prot accession (e.g., `AF-P04637-F1-model_v4.pdb`).
2. **Ref_AA validation (Step 8)**: We verify that the amino acid in the ClinVar record matches the actual amino acid in the canonical protein sequence.

---

### Why UniProt's Own API, Not a Third-Party Library

We previously used `mygene` (MyGene.info), which is a convenient third-party aggregator. However, for a research paper it has two weaknesses:
- It is a **secondary source** - it caches data from UniProt, NCBI, and Ensembl and may lag behind the primary databases after updates.
- It cannot be cited as an authoritative primary source in the Methods section.

We now use `https://rest.uniprot.org/idmapping` directly, which is:
- The **primary authoritative source** - the same data AlphaFold2 and AlphaMissense were built on.
- Directly citable: *"Gene symbols were mapped to canonical UniProt Swiss-Prot accessions using the UniProt REST API ID Mapping service (UniProt Consortium, 2023)."*
- Programmatic and reproducible - no manual download step required.

**Reference:** UniProt Consortium, *Nucleic Acids Research*, 2023 - [doi:10.1093/nar/gkac1052](https://doi.org/10.1093/nar/gkac1052)

---

### Why Swiss-Prot Only - No TrEMBL Fallback

UniProt has two tiers:

| Tier | Description | AlphaFold coverage |
|---|---|---|
| **Swiss-Prot** | Manually reviewed, canonical, curated | ✅ Full coverage |
| **TrEMBL** | Computationally annotated, unreviewed | ❌ Not covered |

AlphaFold EBI builds its entire structure database exclusively from Swiss-Prot canonical sequences (Jumper et al., *Nature* 2021, [doi:10.1038/s41586-021-03819-2](https://doi.org/10.1038/s41586-021-03819-2)). A TrEMBL accession returns 404 on the AlphaFold API. We map to `UniProtKB-Swiss-Prot` explicitly so only reviewed entries are returned.

---

### Biological Deduplication After Mapping
After obtaining UniProt IDs, we apply a second deduplication on `(UniProt_ID, Pos, Alt_AA)` to remove variants that are biologically identical but arrived under different ClinVar `VariationID`s. See Step 6c for the rationale.

In [36]:
import time, re

def map_genesymbols_to_uniprot(gene_list):
    """
    Map gene symbols to Swiss-Prot UniProt IDs using UniProt's official
    programmatic ID Mapping API (https://www.uniprot.org/id-mapping).

    This is the same service used by the UniProt website's 'Map IDs' tool
    and is the primary authoritative source used by AlphaMissense and gMVP.
    Using 'Gene_Name → UniProtKB-Swiss-Prot' ensures we only get manually
    reviewed canonical sequences — the same sequences AlphaFold was built on.
    """
    # Step 1: Submit the mapping job
    submit = requests.post(
        "https://rest.uniprot.org/idmapping/run",
        data={
            "from":  "Gene_Name",
            "to":    "UniProtKB-Swiss-Prot",  # Swiss-Prot only — no TrEMBL
            "ids":   ",".join(gene_list),
            "taxId": "9606"                   # Homo sapiens only
        }
    )
    submit.raise_for_status()
    job_id = submit.json()["jobId"]
    print(f"  Job submitted: {job_id}")

    # Step 2: Poll until complete
    status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
    for attempt in range(60):          # timeout after ~3 minutes
        status = requests.get(status_url).json()
        if "results" in status or "failedIds" in status:
            break
        time.sleep(3)
    else:
        raise TimeoutError(f"UniProt mapping job {job_id} did not complete in time.")

    # Step 3: Fetch paginated results (UniProt returns max 500 per page)
    results_url = (
        f"https://rest.uniprot.org/idmapping/uniprotkb/results/{job_id}"
        f"?format=tsv&fields=accession&size=500"
    )
    gene_to_uniprot = {}
    while results_url:
        r = requests.get(results_url)
        r.raise_for_status()
        lines = r.text.strip().split("\n")[1:]  # skip TSV header row
        for line in lines:
            if "\t" in line:
                parts = line.split("\t")
                gene_sym   = parts[0]
                uniprot_id = parts[1]
                # Keep only the first result per gene (canonical entry)
                if gene_sym not in gene_to_uniprot:
                    gene_to_uniprot[gene_sym] = uniprot_id
        # Follow 'next' pagination link if present
        results_url = None
        if "Link" in r.headers:
            match = re.search(r'<(.+?)>; rel="next"', r.headers["Link"])
            if match:
                results_url = match.group(1)

    return gene_to_uniprot


unique_genes = df['GeneSymbol'].unique().tolist()
print(f"Mapping {len(unique_genes):,} gene symbols → Swiss-Prot via UniProt ID Mapping API...")
print("(You can verify this manually at https://www.uniprot.org/id-mapping)\n")

# UniProt ID Mapping accepts up to 100,000 IDs per job; we batch at 5,000 for stability
gene_to_uniprot = {}
BATCH = 5000
for i in range(0, len(unique_genes), BATCH):
    batch = unique_genes[i : i + BATCH]
    print(f"Batch {i // BATCH + 1} / {-(-len(unique_genes) // BATCH)} ({len(batch):,} genes)...")
    result = map_genesymbols_to_uniprot(batch)
    gene_to_uniprot.update(result)
    print(f"  Mapped: {len(result):,}")

print(f"\nTotal genes mapped to Swiss-Prot: {len(gene_to_uniprot):,} / {len(unique_genes):,}")
print(f"Genes with no Swiss-Prot entry (dropped): {len(unique_genes) - len(gene_to_uniprot):,}")

df['UniProt_ID'] = df['GeneSymbol'].map(gene_to_uniprot)
df = df.dropna(subset=['UniProt_ID']).copy()
print(f"\nAfter UniProt Swiss-Prot mapping: {len(df):,} variants")

# Biological deduplication — same protein + position + alt AA = same biochemical event.
# Different VariationIDs can represent the same mutation submitted under different
# transcript accessions. Deduplicate to prevent train/test contamination.
before = len(df)
df = df.drop_duplicates(subset=['UniProt_ID', 'Pos', 'Alt_AA']).copy()
print(f"After biological dedup (UniProt_ID + Pos + Alt_AA): {before:,} → {len(df):,} (removed {before - len(df):,})")
print(f"  Pathogenic (1): {(df['Target'] == 1).sum():,}")
print(f"  Benign     (0): {(df['Target'] == 0).sum():,}")

Mapping 7,226 gene symbols → Swiss-Prot via UniProt ID Mapping API...
(You can verify this manually at https://www.uniprot.org/id-mapping)

Batch 1 / 2 (5,000 genes)...
  Job submitted: kf9wCYA7kh
  Mapped: 4,999
Batch 2 / 2 (2,226 genes)...
  Job submitted: lYGZM3AhF1
  Mapped: 2,225

Total genes mapped to Swiss-Prot: 7,224 / 7,226
Genes with no Swiss-Prot entry (dropped): 2

After UniProt Swiss-Prot mapping: 62,199 variants
After biological dedup (UniProt_ID + Pos + Alt_AA): 62,199 → 61,847 (removed 352)
  Pathogenic (1): 23,392
  Benign     (0): 38,455


## 8. Reference Amino Acid Validation & Sequence Length Constraint

This is one of the most rigorous quality-control steps in the pipeline. It verifies that the ClinVar → UniProt → AlphaFold mapping chain is **physically correct at the residue level** for every single variant before it enters model training.

---

### Why Ref_AA Validation is Essential

ClinVar variants use transcript-based coordinates (e.g., position 1699 in transcript `NM_007294.4`). UniProt uses canonical protein sequences. These can disagree when:

- The ClinVar entry annotates a **non-canonical isoform** of the gene whose residue numbering differs from the canonical sequence.
- The gene has undergone **transcript version updates** that shifted position numbering.
- There is a **sequence discrepancy** between the submitted transcript and the Swiss-Prot canonical sequence.

If we skip this check, we build GNN graphs where the "mutant node" is placed at the **wrong position in the protein** - we would be training on structurally corrupted data. The model may converge but it would learn meaningless patterns.

**Validation logic:** We download the canonical FASTA sequence for each UniProt ID from the UniProt REST API and confirm that the amino acid at 1-indexed position `Pos` matches `Ref_AA` extracted from the ClinVar Name field. Variants that fail are labelled with a `Validation_Reason` and excluded.

This coordinate validation step is a domain standard adopted by:
- **gMVP** (Ye et al., *Nature Communications* 2023) - [doi:10.1038/s41467-023-42940-8](https://doi.org/10.1038/s41467-023-42940-8)
- **AlphaMissense** (Cheng et al., *Science* 2023) - [doi:10.1126/science.adg7492](https://doi.org/10.1126/science.adg7492)

---

### Sequence Length Constraint (>2700 AA Excluded)

AlphaFold EBI provides pre-computed structures for the vast majority of the human proteome. However, proteins exceeding ~2700 amino acids are often split into overlapping **fragments** in the AlphaFold database (e.g., `F1`, `F2`, `F3`). Mapping a variant at position 3500 to a fragment-stitched structure requires additional logic to determine which fragment covers that position and how to reconcile PAE values across fragment boundaries. We exclude these proteins in Phase 1 to avoid ambiguous structure assignments in Phase 2.

---

### API Pagination - `size=500` Required

> ⚠️ **Implementation note:** The UniProt REST API defaults to **25 results per page**. Without an explicit `size` parameter, a batch of 200 UniProt IDs returns only the first 25 sequences. The remaining 175 proteins get `"Missing Sequence"` in validation and all their variants are silently discarded.

We set `size=500` in every request URL to ensure complete batch retrieval.

---

### Error Handling - No Silent Failures

Failed API batches are logged with a warning rather than silently skipped. A silent `except: pass` makes it impossible to distinguish between "this protein genuinely has no sequence" and "this batch failed due to a transient network error." Logged warnings allow you to identify and re-fetch any missing sequences before finalising the dataset.

In [37]:
aa_map = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

unique_uniprots = df['UniProt_ID'].unique()
print(f"Fetching sequences for {len(unique_uniprots):,} unique proteins from UniProt REST API...")

# --- Sanity check: test one accession before running the full loop ---
# This immediately tells us if the API is reachable and the response format is correct.
test_id = unique_uniprots[0]
test_url = f"https://rest.uniprot.org/uniprotkb/{test_id}.fasta"
test_resp = requests.get(test_url, timeout=30)
print(f"\nSanity check ({test_id}): HTTP {test_resp.status_code}")
if test_resp.status_code == 200 and test_resp.text.strip():
    print(f"Response preview:\n{test_resp.text[:200]}")
else:
    print(f"WARNING: Sanity check failed. Response body: {test_resp.text[:300]}")

# --- Fetch one sequence per protein using the direct FASTA endpoint ---
# We use /uniprotkb/{id}.fasta instead of a batch search query because:
#   1. The batch search URL becomes extremely long for 200+ accessions and can
#      silently fail or be rejected without an error status code.
#   2. The direct endpoint uses the UniProt ID as the URL key, so no FASTA
#      header parsing is needed — the key is always the ID we queried.
#   3. It is the approach recommended in the UniProt REST API documentation.
#
# Rate limit: UniProt allows ~20 requests/second for anonymous users.
# At 50 ms delay we stay well within that limit.

session = requests.Session()
uniprot_seqs = {}
failed_ids = []

for uid in tqdm(unique_uniprots):
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    try:
        response = session.get(url, timeout=30)
        if response.status_code == 200 and response.text.strip():
            record = next(SeqIO.parse(io.StringIO(response.text), "fasta"))
            uniprot_seqs[uid] = str(record.seq)
        else:
            failed_ids.append(uid)
    except Exception as e:
        failed_ids.append(uid)
        print(f"WARNING: {uid} failed — {e}")
    time.sleep(0.05)

print(f"\nSequences fetched : {len(uniprot_seqs):,} / {len(unique_uniprots):,}")
print(f"Failed / not found: {len(failed_ids):,}")
if failed_ids:
    print(f"First 5 failed IDs: {failed_ids[:5]}")

# --- Validate every variant against its canonical sequence ---
def validate_variant(row):
    seq = uniprot_seqs.get(row['UniProt_ID'])
    if not seq:
        return False, "Missing Sequence"
    if len(seq) > 2700:
        return False, "Too Long (>2700 AA)"
    pos = int(row['Pos'])
    if pos < 1 or pos > len(seq):
        return False, "Position Out of Bounds"
    expected_1letter = aa_map.get(row['Ref_AA'])
    if expected_1letter is None:
        return False, f"Unknown Ref_AA: {row['Ref_AA']}"
    actual_aa = seq[pos - 1]   # UniProt sequences are 1-indexed; Python strings are 0-indexed
    if expected_1letter == actual_aa:
        return True, "Valid"
    return False, f"Mismatch: expected {expected_1letter}, got {actual_aa}"

print("\nValidating reference amino acids against canonical UniProt sequences...")
tqdm.pandas()
validation_results = df.progress_apply(validate_variant, axis=1)
df['Is_Valid']          = [r[0] for r in validation_results]
df['Validation_Reason'] = [r[1] for r in validation_results]

print("\n=== Validation Summary ===")
print(df['Validation_Reason'].value_counts().to_string())

final_df = df[df['Is_Valid'] == True].copy()

print(f"\n=== Final Dataset ===")
print(f"Total validated variants : {len(final_df):,}")
print(f"  Pathogenic (1)         : {(final_df['Target'] == 1).sum():,}")
print(f"  Benign     (0)         : {(final_df['Target'] == 0).sum():,}")
ratio = (final_df['Target'] == 0).sum() / max((final_df['Target'] == 1).sum(), 1)
print(f"  Class ratio            : {ratio:.1f}:1 (benign:pathogenic)")
print(f"  Unique proteins        : {final_df['UniProt_ID'].nunique():,}")

Fetching sequences for 7,175 unique proteins from UniProt REST API...

Sanity check (Q30201): HTTP 200
Response preview:
>sp|Q30201|HFE_HUMAN Hereditary hemochromatosis protein OS=Homo sapiens OX=9606 GN=HFE PE=1 SV=1
MGPRARPALLLLMLLQTAVLQGRLLRSHSLHYLFMGASEQDLGLSLFEALGYVDDQLFVF
YDHESRRVEPRTPWVSSRISSQMWLQLSQSLKGWDHMFTVDF


  0%|          | 0/7175 [00:00<?, ?it/s]


Sequences fetched : 7,175 / 7,175
Failed / not found: 0

Validating reference amino acids against canonical UniProt sequences...


  0%|          | 0/61847 [00:00<?, ?it/s]


=== Validation Summary ===
Validation_Reason
Valid                          49820
Too Long (>2700 AA)             8381
Position Out of Bounds           588
Mismatch: expected R, got L       57
Mismatch: expected R, got K       56
Mismatch: expected R, got E       48
Mismatch: expected R, got P       47
Mismatch: expected R, got A       43
Mismatch: expected R, got S       41
Mismatch: expected G, got L       40
Mismatch: expected R, got G       38
Mismatch: expected R, got Q       38
Mismatch: expected A, got L       35
Mismatch: expected V, got L       32
Mismatch: expected R, got I       31
Mismatch: expected P, got S       31
Mismatch: expected R, got V       28
Mismatch: expected P, got K       28
Mismatch: expected A, got S       27
Mismatch: expected A, got K       26
Mismatch: expected A, got P       26
Mismatch: expected G, got A       26
Mismatch: expected P, got G       25
Mismatch: expected P, got E       24
Mismatch: expected R, got N       24
Mismatch: expected A, got V  

## 9. Save Final Dataset

Save the validated, cleaned dataset to CSV for Phase 2 (AlphaFold structure download and graph construction).

**Columns passed forward:**

| Column | Used for |
|---|---|
| `VariationID` | Unique identifier, traceability |
| `GeneSymbol` | Human-readable gene name |
| `UniProt_ID` | AlphaFold PDB download key |
| `Name` | Original HGVS string, for audit |
| `Target` | Label: 1 = pathogenic, 0 = benign |
| `Pos` | Residue position in canonical sequence (1-indexed) |
| `Ref_AA` | Reference amino acid (3-letter) |
| `Alt_AA` | Alternate amino acid (3-letter) |
| `ReviewStatus` | Confidence level - kept for filtering/stratification in later phases |
| `Is_Valid` | Validation flag - all True in final_df, kept for audit trail |
| `Validation_Reason` | Validation outcome - kept for audit trail |

In [39]:
output_file = "clinvar_missense_filtered.csv"
final_df.to_csv(output_file, index=False)
print(f"Saved to {output_file}")
final_df[['VariationID', 'GeneSymbol', 'UniProt_ID', 'Name', 'Target', 'Pos', 'Ref_AA', 'Alt_AA']].head()

Saved to clinvar_missense_filtered.csv


,VariationID,GeneSymbol,UniProt_ID,Name,Target,Pos,Ref_AA,Alt_AA
33,19,HFE,Q30201,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),1,283,Gln,Pro
55,30,HOGA1,Q86XE5,NM_138413.4(HOGA1):c.860G>T (p.Gly287Val),1,287,Gly,Val
57,31,HOGA1,Q86XE5,NM_138413.4(HOGA1):c.289C>T (p.Arg97Cys),1,97,Arg,Cys
61,33,HOGA1,Q86XE5,NM_138413.4(HOGA1):c.209G>C (p.Arg70Pro),1,70,Arg,Pro
63,34,HOGA1,Q86XE5,NM_138413.4(HOGA1):c.769T>G (p.Cys257Gly),1,257,Cys,Gly
